# **PARALLEL CHAINS** (Groq Version)

RunnableParallel sends the SAME input to MULTIPLE chains simultaneously
and collects all their outputs into a single dict. Great for fan-out tasks
like generating posts for several platforms at once.

In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


Bro API KEY Variable exists


In [2]:
# TASK 1 — Summarise a movie first; the summary feeds into both parallel branches
# {input} = the movie name typed by the user
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human",  "Please summarize the movie in brief : {input}")
])


In [3]:
# TASK 2 — LLM for the summarizer step
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


In [4]:
# TASK 3 — Parse summary to plain string
str_parser = StrOutputParser()


In [5]:
# TASK 4 — Format bridge (same pattern as previous notebook)
# Wraps the summary string into {"text": summary} so downstream prompts can use {text}
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)


### **Parallel Branch 1 — LinkedIn post**

A standalone chain that takes `{text}` (the summary) and writes a LinkedIn post.

In [6]:
# LinkedIn branch: straightforward 3-step chain
# Input expected: a dict with key 'text' (provided by RunnableParallel)
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human",  "Create a post for the following text for LinkedIn: {text}")
])

llm_groq     = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
str_parser   = StrOutputParser()

# This is a complete mini-chain: prompt → llm → string
chain_linkedin = linkedin_prompt | llm_groq | str_parser


### **Parallel Branch 2 — Instagram post**

Defined as a function + RunnableLambda. This pattern is useful when your branch
has complex logic or needs to build its own chain dynamically.

In [7]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

# RunnableParallel will call this function with the SAME dict that chain_linkedin gets
# The dict looks like: {"text": "<movie summary>"}
def insta_chain(text: dict):
    # Extract the summary string from the dict
    text = text["text"]

    # Build a fresh mini-chain internally
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        ("human",  "Create a post for the following text for Instagram: {text}")
    ])
    llm_groq_inner = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    str_parser_inner = StrOutputParser()

    chain_insta = insta_prompt | llm_groq_inner | str_parser_inner

    # .invoke(text) — note: passing the string directly since {text} is a single-var template
    result = chain_insta.invoke(text)
    return result

# Wrap the function so it's usable inside a | chain
insta_chain_runnable = RunnableLambda(insta_chain)


### **Final Orchestration — putting it all together**

In [8]:
# The full pipeline:
#  prompt_template           → produce movie-summary prompt
#  llm_groq                  → generate summary (AIMessage)
#  str_parser                → extract summary string
#  dictionary_maker_runnable → wrap to {"text": summary}
#  RunnableParallel(...)     → SPLIT: send {"text":...} to BOTH branches simultaneously
#                              returns {"branches": {"linkedin": "...", "instagram": "..."}}
final_chain = (
    prompt_template
    | llm_groq
    | str_parser
    | dictionary_maker_runnable
    | RunnableParallel(branches={"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)


In [9]:
# Run the full chain for 'KGF'
# Output: {'branches': {'linkedin': '...post...', 'instagram': '...post...'}}
final_chain.invoke("KGF")


{'branches': {'linkedin': "Here's a LinkedIn-style post based on the provided text:\n\n**Leadership Lessons from the Big Screen: KGF**\n\nAs professionals, we often draw inspiration from unexpected sources. For me, the 2018 Indian action film KGF (Kolar Gold Fields) is a great example of perseverance, strategic thinking, and leadership.\n\nThe movie's protagonist, Rocky, played by Yash, embodies the spirit of entrepreneurship and risk-taking. He leaves behind the comforts of his life in Mumbai to fulfill his mother's dying wish and build a better future for himself. His journey is a testament to the power of determination and hard work.\n\nAs Rocky navigates the challenges of the gold mafia and battles against the ruthless owner, Garuda, he demonstrates key leadership skills:\n\n **Visionary thinking**: Rocky has a clear vision of what he wants to achieve and is willing to take calculated risks to get there.\n **Strategic planning**: He forms alliances, gathers resources, and outsmarts

## **Chain as a Runnable — flattening nested output**

The parallel output has an extra 'branches' wrapper. We add one more step to clean that up.

In [10]:
# TASK — Beautify: flatten the nested dict so callers get a clean top-level result
def beautify(final_response: dict) -> dict:
    # final_response looks like: {"branches": {"linkedin": "...", "instagram": "..."}}
    linkedin_response  = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']
    # Return a flat dict — much easier to use downstream
    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)

# Pipe final_chain into the beautifier
# The | here appends one more step to an already-built chain — chains are composable!
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")
# Output: {'linkedin': '...post...', 'instagram': '...post...'}


{'linkedin': 'Here\'s a LinkedIn-style post based on the provided text:\n\n**Leadership Lessons from the Unlikeliest of Places: "Pushpa: The Rise"**\n\nAs professionals, we often draw inspiration from unexpected sources. The 2021 Indian film "Pushpa: The Rise" is a powerful example of how ambition, determination, and strategic risk-taking can lead to success, even in the face of adversity.\n\nThe movie\'s protagonist, Pushpa Raj, is a testament to the human spirit\'s capacity for growth and transformation. From humble beginnings as a poor laborer, he rises to become a dominant force in the red sandalwood smuggling business, navigating complex webs of relationships, rival gangs, and corrupt systems along the way.\n\nSo, what can we learn from Pushpa\'s journey?\n\n **Ambition and perseverance can take you far**: Pushpa\'s unwavering dedication to his goals is a reminder that success often requires sacrifice, hard work, and a willingness to take calculated risks.\n\n **Power dynamics are